# Sigma extrapolation test

One-off notebook. `DataCalculator.interpolate_to_sigma` interpolates each profile from the ERA5 pressure levels (500–1000 hPa) to σ = 0.50, 0.55, …, 1.00, where the target pressure is σ·ps. When σ·ps falls outside 500–1000 hPa, the index is clamped but the interpolation weight is not, so the value is extrapolated linearly from the two nearest levels:

- σ·ps < 500 hPa (ps < 1000 hPa at σ = 0.5): extrapolated above 500 hPa from the 500–550 hPa gradient.
- σ·ps > 1000 hPa (ps > 1000 hPa at σ = 1.0): extrapolated below 1000 hPa from the 975–1000 hPa gradient.

The proposed fix clamps the weight to [0, 1], so out-of-range targets take the value at 500 or 1000 hPa. This notebook (1) counts how often each σ level is out of range over all years, and (2) compares the current and clamped interpolation for one year. Reads raw files only and writes nothing.

In [ ]:
import sys
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from IPython.display import display
sys.path.insert(0,'..')
from scripts.utils import Config
from scripts.data.classes import DataCalculator
warnings.filterwarnings('ignore')

In [ ]:
CONFIG    = Config()
YEAR      = 2018
SIGS      = np.arange(0.5,1.05,0.05,dtype=np.float32)
LEVMIN,LEVMAX = CONFIG.levrange
FIELDVARS = ['rh','thetae','thetaestar']
VARLABELS = {'rh':'RH (%)','thetae':'θe (K)','thetaestar':'θe* (K)'}

In [ ]:
def calc_targets(ps,sigs):
    '''
    Purpose: Target pressures σ·ps for every sigma level.
    Args:
    - ps (xr.DataArray): surface pressure (hPa) with dims (lat, lon, time)
    - sigs (np.ndarray): sigma levels
    Returns:
    - np.ndarray: target pressures with shape (lat, lon, sig, time)
    '''
    return sigs[None,None,:,None]*ps.transpose('lat','lon','time').values[:,:,None,:]

def interpolate_to_sigma(da,ps,sigs,clamp):
    '''
    Purpose: Copy of DataCalculator.interpolate_to_sigma in float64, optionally clamping the weight to [0, 1].
    Args:
    - da (xr.DataArray): field with 'lev'
    - ps (xr.DataArray): surface pressure (hPa)
    - sigs (np.ndarray): ascending sigma levels
    - clamp (bool): if True, hold the nearest level's value outside 500–1000 hPa instead of extrapolating
    Returns:
    - xr.DataArray: field with dims (lat, lon, sig, time)
    '''
    da      = da.transpose('lat','lon','lev','time')
    levs    = da.lev.values
    ptarget = calc_targets(ps,sigs)
    upper   = np.clip(np.searchsorted(levs,ptarget,side='right'),1,len(levs)-1)
    lower   = upper-1
    weight  = (ptarget-levs[lower])/(levs[upper]-levs[lower])
    if clamp:
        weight = np.clip(weight,0.0,1.0)
    result  = (1.0-weight)*np.take_along_axis(da.values,lower,axis=2)+weight*np.take_along_axis(da.values,upper,axis=2)
    result  = np.where(np.isfinite(ptarget)&(ptarget>0),result,np.nan)
    return xr.DataArray(result,dims=('lat','lon','sig','time'),coords={'lat':da.lat,'lon':da.lon,'sig':sigs,'time':da.time},name=da.name)

def plot_column(name,ax):
    '''
    Purpose: Plot the column with the largest current value of a variable: ERA5 pressure-level values, current
        (extrapolated) sigma values, and clamped sigma values, all against pressure.
    Args:
    - name (str): variable name
    - ax (matplotlib.axes.Axes): axes to plot on
    '''
    values = current[name].values
    ilat,ilon,_,itime = np.unravel_index(np.nanargmax(values),values.shape)
    column = dict(lat=ilat,lon=ilon,time=itime)
    pscol  = float(psyear.isel(**column))
    ax.plot(profiles[name].isel(**column).values,profiles[name].lev.values,'ko-',ms=3,label='ERA5 pressure levels')
    ax.plot(current[name].isel(**column).values,SIGS*pscol,'r^--',ms=4,label='σ levels, current')
    ax.plot(clamped[name].isel(**column).values,SIGS*pscol,'bs:',ms=3,label='σ levels, clamped')
    for lev in (LEVMIN,LEVMAX):
        ax.axhline(lev,color='gray',lw=0.5,ls='--')
    ax.axhline(pscol,color='brown',lw=0.8,label='surface pressure')
    ax.invert_yaxis()
    ax.set_xlabel(VARLABELS[name])
    ax.set_ylabel('Pressure (hPa)')
    ax.set_title(f'{float(psyear.lat[ilat]):.0f}°N {float(psyear.lon[ilon]):.0f}°E, {pd.Timestamp(psyear.time.values[itime]):%Y-%m-%d %H:%M}',fontsize=9)
    ax.legend(fontsize=7)

## 1. How often is each σ level outside 500–1000 hPa? (all years, all hours)

In [ ]:
calculator = DataCalculator(
    author=CONFIG.author,
    email=CONFIG.email,
    filedir=CONFIG.rawdir,
    savedir=None,
    latrange=CONFIG.latrange,
    lonrange=CONFIG.lonrange)
ps = calculator.regrid(calculator.retrieve('ERA5_surface_pressure').load().astype(np.float64)).transpose('lat','lon','time')
with calculator.retrieve('ERA5_air_temperature') as tref:
    LEVS = np.sort(tref.lev.values.astype(np.float64))
print(f'Pressure levels (hPa): {LEVS.tolist()}')
print(f'Surface pressure on the 1° grid: {float(ps.min()):.1f} – {float(ps.max()):.1f} hPa')

In [ ]:
rows = []
for sig in SIGS:
    target = float(sig)*ps.values
    above  = target<LEVMIN
    below  = target>LEVMAX
    rows.append({
        'σ':round(float(sig),2),
        'above 500 hPa (% samples)':100*above.mean(),
        'below 1000 hPa (% samples)':100*below.mean(),
        'grid cells ever affected':int((above|below).any(axis=-1).sum()),
        'most negative weight':float(((target[above]-LEVS[0])/(LEVS[1]-LEVS[0])).min()) if above.any() else 0.0,
        'largest weight':float(((target[below]-LEVS[-2])/(LEVS[-1]-LEVS[-2])).max()) if below.any() else 1.0})
display(pd.DataFrame(rows).set_index('σ').round(3))
print('Weights in [0, 1] interpolate; a weight of −2 extends the 500–550 hPa trend two layer widths above 500 hPa.')

In [ ]:
fig,axs = plt.subplots(1,3,figsize=(16,4))
ps.min('time').plot(ax=axs[0],cmap='viridis')
axs[0].set_title('Minimum surface pressure (hPa)')
(100*(ps<LEVMAX).mean('time')).plot(ax=axs[1],cmap='Reds',vmin=0,vmax=100)
axs[1].set_title('% of hours with σ = 0.5 above 500 hPa (ps < 1000)')
(100*(ps>LEVMAX).mean('time')).plot(ax=axs[2],cmap='Blues',vmin=0,vmax=100)
axs[2].set_title('% of hours with σ = 1.0 below 1000 hPa (ps > 1000)')
plt.tight_layout()
plt.show()

## 2. Current vs clamped interpolation for one year

In [ ]:
t = calculator.retrieve('ERA5_air_temperature')
q = calculator.retrieve('ERA5_specific_humidity')
t = calculator.regrid(t.sel(time=t.time.dt.year==YEAR).load().astype(np.float64))
q = calculator.regrid(q.sel(time=q.time.dt.year==YEAR).load().astype(np.float64))
psyear = ps.sel(time=ps.time.dt.year==YEAR)
print(f'Timestamps aligned: {np.array_equal(t.time.values,psyear.time.values) and np.array_equal(q.time.values,psyear.time.values)}')
p = calculator.create_p_array(q)
profiles = {
    'rh':calculator.calc_rh(p,t,q),
    'thetae':calculator.calc_thetae(p,t,q),
    'thetaestar':calculator.calc_thetae(p,t)}
current = {name:interpolate_to_sigma(da,psyear,SIGS,clamp=False) for name,da in profiles.items()}
clamped = {name:interpolate_to_sigma(da,psyear,SIGS,clamp=True) for name,da in profiles.items()}

In [ ]:
rows = []
for name in FIELDVARS:
    for method,da in [('pressure levels',profiles[name]),('σ, current',current[name]),('σ, clamped',clamped[name])]:
        values = da.values
        rows.append({'variable':name,'method':method,'min':np.nanmin(values),'max':np.nanmax(values),'mean':np.nanmean(values),'std':np.nanstd(values)})
display(pd.DataFrame(rows).set_index(['variable','method']).round(3))
for name in FIELDVARS:
    changed = np.isfinite(current[name].values)&(current[name].values!=clamped[name].values)
    print(f'{name}: {100*changed.mean():.3f}% of σ values change with clamping')
print(f'RH > 100%: current {int((current["rh"]>100).sum())} values, clamped {int((clamped["rh"]>100).sum())}')

In [ ]:
fig,axs = plt.subplots(1,3,figsize=(16,5))
for name,ax in zip(FIELDVARS,axs):
    plot_column(name,ax)
fig.suptitle(f'Columns with the largest current value, {YEAR}')
plt.tight_layout()
plt.show()

In [ ]:
fig,axs = plt.subplots(2,3,figsize=(16,8))
for j,name in enumerate(FIELDVARS):
    vmin = float(min(current[name].isel(sig=0).max('time').min(),clamped[name].isel(sig=0).max('time').min()))
    vmax = float(current[name].isel(sig=0).max('time').max())
    for i,(method,data) in enumerate([('current',current),('clamped',clamped)]):
        data[name].isel(sig=0).max('time').plot(ax=axs[i,j],vmin=vmin,vmax=vmax,cmap='magma')
        axs[i,j].set_title(f'Max {VARLABELS[name]} at σ = 0.5, {method}',fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
fig,axs = plt.subplots(1,3,figsize=(16,4))
for name,ax in zip(FIELDVARS,axs):
    meancurrent = current[name].mean(['lat','lon','time'])
    meanclamped = clamped[name].mean(['lat','lon','time'])
    ax.plot(meancurrent.values,SIGS,'r^--',label='current')
    ax.plot(meanclamped.values,SIGS,'bs:',label='clamped')
    ax.invert_yaxis()
    ax.set_xlabel(f'Domain-mean {VARLABELS[name]}')
    ax.set_ylabel('σ')
    ax.legend(fontsize=8)
fig.suptitle(f'Domain-mean profiles, {YEAR}')
plt.tight_layout()
plt.show()